# PhysicsConv3D: Supervised Pretraining – Spektren und Parameter-Maps

Das Notebook lädt einen echten WALINET-bereinigten 7T-Datensatz, wendet dieselbe subjectweise FID-Normalisierung und FFT wie das Training an und führt einen vollständigen Z-Slice durch das baselinefreie PhysicsConv3D-Modell. Auswertung und Maps werden auf die Brain-Mask beschränkt. Geladen wird der Run mit supervised auf den klassischen forD-Parameter-Maps trainierten Gewichten.

In [ ]:
from pathlib import Path
import importlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_project_root(start: Path, name: str) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if candidate.name == name and (candidate / 'src').is_dir():
            return candidate
        sibling = candidate / name
        if (sibling / 'src').is_dir():
            return sibling
    raise FileNotFoundError(name)


ROOT = find_project_root(Path.cwd(), 'Denoising')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from denoising.config.build import build_config
from denoising.config.load import load_yaml

# A notebook kernel may still hold the model implementation from an older
# training/evaluation run. Reload in dependency order so inference always
# uses the current scaled parameterization from the source tree.
import denoising.models.physics.parameterization as parameterization_module
import denoising.models.physics.physics_conv3d as physics_conv3d_module
import denoising.models.factory as model_factory_module
importlib.reload(parameterization_module)
importlib.reload(physics_conv3d_module)
importlib.reload(model_factory_module)
build_model = model_factory_module.build_model

print('Denoising:', ROOT)

## Einstellungen

Das Notebook verwendet immer `last.pt`, damit während des Trainings der neueste vollständig gespeicherte Epochenstand betrachtet wird.

In [ ]:
RUN_NAME = 'PhysicsConv3D_7T_supervised_forD'
CHECKPOINT_NAME = 'last.pt'  # immer der neueste abgeschlossene Epochenstand
SUBJECT = 'Vol1_Brisbane'
Z_SLICE = 17
GPU_NUMBER = 0

# Der komplette 64x64-Slice wird als ein Patch ausgewertet. Dadurch gibt
# es keine periodischen Sliding-Window-/Padding-Artefakte in den Maps.
PATCH_SIZE = 64
PATCH_STRIDE = 64
INFERENCE_BATCH_SIZE = 1

# Spektrum an diesem Voxel; falls es außerhalb der Maske liegt, wird
# automatisch das maskierte Voxel gewählt, das dem Bildzentrum am nächsten ist.
VOXEL_XY = (32, 32)

DEVICE = torch.device(f'cuda:{GPU_NUMBER}' if torch.cuda.is_available() else 'cpu')
RUN_DIR = ROOT / 'trained_models' / RUN_NAME
CONFIG_PATH = RUN_DIR / 'pretrain_physics_supervised_7T.yaml'
CHECKPOINT_PATH = RUN_DIR / 'checkpoints' / CHECKPOINT_NAME
if not CONFIG_PATH.is_file():
    raise FileNotFoundError(f'Run config not found: {CONFIG_PATH}')
if not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError(
        f'No supervised checkpoint yet: {CHECKPOINT_PATH}. '
        'Wait until the first training epoch has completed.'
    )
print('Device:    ', DEVICE)
print('Config:    ', CONFIG_PATH)
print('Checkpoint:', CHECKPOINT_PATH)

## Config, Subject und Modell laden

In [ ]:
cfg = build_config(load_yaml(CONFIG_PATH))
if cfg.run.name != RUN_NAME:
    raise RuntimeError(f'Config run {cfg.run.name!r} does not match {RUN_NAME!r}.')
if cfg.data.spatial_mask_filename is None:
    raise RuntimeError('This checkpoint config has no spatial brain mask.')
data_path = ROOT / cfg.data.base_dir / SUBJECT / cfg.data.data_filename
mask_path = ROOT / cfg.data.base_dir / SUBJECT / cfg.data.spatial_mask_filename

fid_volume = np.load(data_path).astype(np.complex64, copy=False)
brain_mask = np.load(mask_path).astype(bool)
if fid_volume.ndim != 4 or fid_volume.shape[-1] != 840:
    raise ValueError(f'Unexpected data shape: {fid_volume.shape}')
if not 0 <= Z_SLICE < fid_volume.shape[2]:
    raise IndexError(f'Z_SLICE must be in [0, {fid_volume.shape[2] - 1}]')

# Identical to load_and_preprocess_data: one scale per complete subject,
# calculated in FID domain before FFT.
normalization_scale = float(np.max(np.abs(fid_volume))) if cfg.data.normalization else 1.0
normalized_fids = fid_volume / normalization_scale if normalization_scale > 0 else fid_volume
spectra_volume = np.fft.fftshift(
    np.fft.fft(normalized_fids, axis=3), axes=3
).astype(np.complex64)
input_slice = spectra_volume[:, :, Z_SLICE, :]
mask_slice = brain_mask[:, :, Z_SLICE]

sample_shape = (2, PATCH_SIZE, PATCH_SIZE, input_slice.shape[-1])
model = build_model(cfg, sample_shape).to(DEVICE)
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint['model_state'], strict=True)
model.eval()

print('Data:       ', data_path)
print('FID shape:  ', fid_volume.shape)
print('Slice shape:', input_slice.shape)
print('FID scale:  ', normalization_scale)
print('Epoch:      ', checkpoint.get('epoch'))
print('Val loss:   ', checkpoint.get('val_loss'))
print('Basis:      ', model.physical_decoder.basis_fids.shape)

## Patchweise Inference

Überlappende Patches werden gleich gewichtet gemittelt. Neben dem rekonstruierten Spektrum werden die bereits physikalisch transformierten Parameter gesammelt.

In [ ]:
def patch_starts(size, patch_size, stride):
    starts = list(range(0, max(size - patch_size + 1, 1), stride))
    last = max(size - patch_size, 0)
    if not starts or starts[-1] != last:
        starts.append(last)
    return starts


size_x, size_y, n_frequency = input_slice.shape
x_starts = patch_starts(size_x, PATCH_SIZE, PATCH_STRIDE)
y_starts = patch_starts(size_y, PATCH_SIZE, PATCH_STRIDE)
locations = [(x, y) for x in x_starts for y in y_starts]
n_basis = model.physical_decoder.n_basis_components

reconstruction_sum = np.zeros_like(input_slice, dtype=np.complex64)
amplitude_sum = np.zeros((size_x, size_y, n_basis), dtype=np.float32)
nuisance_sum = {
    'Frequency shift [Hz]': np.zeros((size_x, size_y), np.float32),
    'Lorentz FWHM [Hz]': np.zeros((size_x, size_y), np.float32),
    'Gaussian FWHM [Hz]': np.zeros((size_x, size_y), np.float32),
    'Phase 0 [rad]': np.zeros((size_x, size_y), np.float32),
    'Phase 1 [rad/Hz]': np.zeros((size_x, size_y), np.float32),
}
weight_sum = np.zeros((size_x, size_y), dtype=np.float32)

with torch.inference_mode():
    for start in range(0, len(locations), INFERENCE_BATCH_SIZE):
        batch_locations = locations[start:start + INFERENCE_BATCH_SIZE]
        patches = [
            input_slice[x:x + PATCH_SIZE, y:y + PATCH_SIZE]
            for x, y in batch_locations
        ]
        complex_batch = np.stack(patches)
        network_batch = torch.from_numpy(
            np.stack((complex_batch.real, complex_batch.imag), axis=1)
        ).to(DEVICE)  # [B, 2, X, Y, F]
        output = model(network_batch, return_parameters=True)
        reconstruction = (
            output.reconstruction[:, 0] + 1j * output.reconstruction[:, 1]
        ).cpu().numpy()
        parameters = output.parameters
        amplitudes = parameters.amplitudes.cpu().numpy()
        nuisance_batches = [
            parameters.frequency_shift_hz.cpu().numpy(),
            parameters.lorentzian_fwhm_hz.cpu().numpy(),
            parameters.gaussian_fwhm_hz.cpu().numpy(),
            parameters.zero_order_phase_radians.cpu().numpy(),
            parameters.first_order_phase_rad_per_hz.cpu().numpy(),
        ]
        for index, (x, y) in enumerate(batch_locations):
            area = np.s_[x:x + PATCH_SIZE, y:y + PATCH_SIZE]
            reconstruction_sum[area] += reconstruction[index]
            amplitude_sum[area] += amplitudes[index]
            for name, values in zip(nuisance_sum, nuisance_batches):
                nuisance_sum[name][area] += values[index]
            weight_sum[area] += 1.0

reconstruction_slice = reconstruction_sum / weight_sum[..., None]
amplitude_maps = amplitude_sum / weight_sum[..., None]
nuisance_maps = {name: values / weight_sum for name, values in nuisance_sum.items()}
residual_slice = input_slice - reconstruction_slice
masked_input = input_slice[mask_slice]
masked_reconstruction = reconstruction_slice[mask_slice]
masked_residual = residual_slice[mask_slice]
input_rms = float(np.sqrt(np.mean(np.abs(masked_input) ** 2)))
reconstruction_rms = float(np.sqrt(np.mean(np.abs(masked_reconstruction) ** 2)))
masked_complex_mse = float(np.mean(np.abs(masked_residual) ** 2))
print(f'Patches: {len(locations)}, inference complete')
print('Reconstruction:', reconstruction_slice.shape)
print('Amplitude maps:', amplitude_maps.shape)
print(f'Brain-mask input RMS:          {input_rms:.6g}')
print(f'Brain-mask reconstruction RMS: {reconstruction_rms:.6g}')
output_to_input_rms = reconstruction_rms / input_rms if input_rms > 0 else np.nan
zero_solution_mse = float(np.mean(np.abs(masked_input) ** 2))
print(f'Brain-mask complex MSE:         {masked_complex_mse:.6g}')
print(f'Zero-solution complex MSE:      {zero_solution_mse:.6g}')
print(f'Output / input RMS:             {output_to_input_rms:.4f}')
if output_to_input_rms < 0.1:
    print('WARNING: Reconstruction energy is below 10% of the input; possible zero collapse.')

## Input, Rekonstruktion und Residuum an einem Voxel

In [ ]:
voxel_x, voxel_y = VOXEL_XY
if not mask_slice[voxel_x, voxel_y]:
    coordinates = np.argwhere(mask_slice)
    center = np.array([size_x / 2, size_y / 2])
    voxel_x, voxel_y = coordinates[np.argmin(np.sum((coordinates - center) ** 2, axis=1))]

input_spectrum = input_slice[voxel_x, voxel_y]
fitted_spectrum = reconstruction_slice[voxel_x, voxel_y]
residual_spectrum = residual_slice[voxel_x, voxel_y]
frequency_hz = np.fft.fftshift(np.fft.fftfreq(
    n_frequency, d=model.physical_decoder.dwell_time_seconds
))

fig, axes = plt.subplots(2, 2, figsize=(18, 10), constrained_layout=True)
axes[0, 0].plot(frequency_hz, input_spectrum.real, label='Input', lw=1.2)
axes[0, 0].plot(frequency_hz, fitted_spectrum.real, label='Physics reconstruction', lw=1.2)
axes[0, 1].plot(frequency_hz, input_spectrum.imag, label='Input', lw=1.2)
axes[0, 1].plot(frequency_hz, fitted_spectrum.imag, label='Physics reconstruction', lw=1.2)
axes[1, 0].plot(frequency_hz, np.abs(input_spectrum), label='Input magnitude')
axes[1, 0].plot(frequency_hz, np.abs(fitted_spectrum), label='Reconstruction magnitude')
axes[1, 1].plot(frequency_hz, residual_spectrum.real, label='Residual real')
axes[1, 1].plot(frequency_hz, residual_spectrum.imag, label='Residual imaginary', alpha=0.8)
titles = ['Real', 'Imaginary', 'Magnitude', 'Input − reconstruction']
for ax, title in zip(axes.flat, titles):
    ax.set_title(title)
    ax.set_xlabel('Frequency offset [Hz]')
    ax.grid(alpha=0.2)
    ax.legend()
fig.suptitle(f'{SUBJECT}, z={Z_SLICE}, voxel=({voxel_x}, {voxel_y}), epoch={checkpoint.get("epoch")}', fontsize=16)
plt.show()

## Amplituden-Maps aller Basisbestandteile

Jede Map verwendet ihre eigene robuste Skala innerhalb der Hirnmaske (1.–99. Perzentil). Angezeigt werden die positiven physikalischen Amplituden (`softplus(raw)`), nicht die internen Raw-Outputs. Die Werte beziehen sich auf die subjectweise normalisierten Trainingsdaten.

In [ ]:
basis_names = tuple(model.basis_names)
n_columns = 5
n_rows = int(np.ceil(len(basis_names) / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(20, 3.6 * n_rows), constrained_layout=True)
for index, (name, ax) in enumerate(zip(basis_names, axes.flat)):
    values = amplitude_maps[..., index]
    inside = values[mask_slice & np.isfinite(values)]
    vmin, vmax = (np.percentile(inside, [1, 99]) if inside.size else (0, 1))
    if vmax <= vmin:
        vmax = vmin + 1e-6
    display = np.where(mask_slice, values, np.nan)
    image = ax.imshow(display.T, origin='lower', cmap='magma', vmin=vmin, vmax=vmax)
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03)
for ax in axes.flat[len(basis_names):]:
    ax.axis('off')
fig.suptitle(f'Baseline-free amplitude maps — {SUBJECT}, z={Z_SLICE}', fontsize=18)
plt.show()

## Frequenz-, Linienbreiten- und Phasen-Maps

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4.5), constrained_layout=True)
for ax, (name, values) in zip(axes, nuisance_maps.items()):
    inside = values[mask_slice & np.isfinite(values)]
    vmin, vmax = (np.percentile(inside, [1, 99]) if inside.size else (0, 1))
    if vmax <= vmin:
        vmax = vmin + 1e-6
    display = np.where(mask_slice, values, np.nan)
    image = ax.imshow(display.T, origin='lower', cmap='coolwarm', vmin=vmin, vmax=vmax)
    ax.set_title(name)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.03)
fig.suptitle(f'Nuisance parameter maps — {SUBJECT}, z={Z_SLICE}', fontsize=17)
plt.show()